# 4.3 Data Augmentation

在 flower_photos 上加入資料增強。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import pathlib

## 1. 載入資料

In [ ]:
url = 'https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz'
data_dir = pathlib.Path(tf.keras.utils.get_file('flower_photos', origin=url, untar=True))
# Some environments extract the archive into an extra nested folder.
if (data_dir / 'flower_photos').exists():
    data_dir = data_dir / 'flower_photos'
img_size = (180, 180)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2, subset='training', seed=42, image_size=img_size, batch_size=batch_size)
val_ds = tf.keras.utils.image_dataset_from_directory(data_dir, validation_split=0.2, subset='validation', seed=42, image_size=img_size, batch_size=batch_size)
num_classes = len(train_ds.class_names)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)

## 2. 建立資料增強 layer

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
])

## 3. 預覽資料增強效果

In [ ]:
for images, labels in train_ds.take(1):
    plt.figure(figsize=(8, 8))
    image = images[0]
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        augmented = data_augmentation(tf.expand_dims(image, 0), training=True)
        plt.imshow(augmented[0].numpy().astype('uint8'))
        plt.axis('off')
    plt.show()

## 4. 建立含資料增強的 CNN

In [ ]:
tf.keras.utils.set_random_seed(42)

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=img_size + (3,)),
    data_augmentation,
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 5. 訓練模型

In [ ]:
history = model.fit(train_ds, validation_data=val_ds, epochs=5)
pd.DataFrame(history.history)[['accuracy', 'val_accuracy']].plot(figsize=(8, 4), title='Accuracy Curve')
plt.show()

## 6. 套用自己的資料

套用到自己的圖片時，augmentation 要符合影像語意。只加入不會改變標籤意義的變化，例如適合一般物體的水平翻轉，不一定適合文字圖片或有固定方向的醫療影像。


## 7. 小結

Data augmentation 可以讓模型在訓練時看到更多合理變化，是小型影像資料集常用的正則化方法。
